## PACOTES 

In [31]:
import os
import time

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import ks_2samp

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    f1_score,
    matthews_corrcoef,
    log_loss,
    confusion_matrix
)

from openTSNE import TSNE

## CONF GERAIS

In [32]:
# BASE
RANK = 1
TARGET_COL_D = "status_fraude"
THRESHOLD_D = 0.50
estado_randomico_D = 42
NOME_HTML_D = f"3d_rank_{RANK}_tsne.html"

# HIPERPARÂMETROS GMM
numero_de_componentes_D = 2
inicializacoes_gausianas_D = 3
tipo_matriz_covariancia_D = "full"
erro_numerico_D = 1e-6

# HIPERPARÂMETROS t-SNE
TSNE_PERPLEXITY_D = 30
TSNE_N_ITER_D = 1500
TSNE_EARLY_EXAGGERATION_ITER_D = 100
TSNE_EARLY_EXAGGERATION_D = 12
TSNE_EXAGGERATION_D = 1
TSNE_LEARNING_RATE_D = "auto"
TSNE_METRIC_D = "euclidean"
TSNE_INITIALIZATION_D = "pca"
TSNE_NEGATIVE_GRADIENT_METHOD_D = "bh"
TSNE_N_JOBS_D = 7
TSNE_RANDOM_STATE_D = estado_randomico_D
TSNE_VERBOSE_D = True


## FUNCAO SCORE 

In [33]:
def calcular_score_final():
    auc_pr_norm = np.clip(auc_pr, 0, 1)
    mcc_norm = (mcc + 1) / 2
    mcc_norm = np.clip(mcc_norm, 0, 1)
    ks_norm = np.clip(ks, 0, 1)
    log_loss_norm = 1 / (1 + ll)
    score = (
        mcc_norm +
        ks_norm +
        log_loss_norm +
        auc_pr_norm
    ) / 4

    return round(float(score), 6)


## FUNCAO T-SNE/ORIGINAL 

In [34]:
def gerar_relatorio_2d(
    RANK=1,
    usar_tsne=True,
    TARGET_COL="status_fraude",
    THRESHOLD=0.50,
    estado_randomico=42,

    numero_de_componentes=2,
    inicializacoes_gausianas=3,
    tipo_matriz_covariancia="full",
    erro_numerico=1e-6,

    TSNE_PERPLEXITY=30,
    TSNE_N_ITER=1000,
    TSNE_EARLY_EXAGGERATION_ITER=100,
    TSNE_EARLY_EXAGGERATION=12,
    TSNE_EXAGGERATION=1,
    TSNE_LEARNING_RATE="auto",
    TSNE_METRIC="euclidean",
    TSNE_INITIALIZATION="pca",
    TSNE_NEGATIVE_GRADIENT_METHOD="bh",
    TSNE_N_JOBS=7,
    TSNE_VERBOSE=True
):

    def calcular_score_final():
        auc_pr_norm = np.clip(auc_pr, 0, 1)

        mcc_norm = (mcc + 1) / 2
        mcc_norm = np.clip(mcc_norm, 0, 1)

        ks_norm = np.clip(ks, 0, 1)

        log_loss_norm = 1 / (1 + ll)

        score = (
            auc_pr_norm +
            mcc_norm +
            ks_norm +
            log_loss_norm
        ) / 4

        return round(float(score), 6)

    tipo_relatorio = "tsne" if usar_tsne else "orig"
    NOME_HTML = f"2d_rank_{RANK}_{tipo_relatorio}.html"

    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()

    HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

    print("Diretório:", BASE_DIR)

    # LOAD RANKING
    df_scores = pd.read_csv("2x2_visu_scores.csv")

    row = df_scores[
        df_scores["Posicao_Rank"] == RANK
    ].iloc[0]

    feature_1 = row["Feature_1"]
    feature_2 = row["Feature_2"]

    print(f"\nRank Selecionado: {RANK}")
    print(f"Features: {feature_1} vs {feature_2}")

    # LOAD DATASET
    df = pd.read_csv("creditcard.csv")

    df_model = df[
        [feature_1, feature_2, TARGET_COL]
    ].dropna().reset_index(drop=True)

    print("\nQuantidade usada:")
    print(df_model[TARGET_COL].value_counts())

    X_original = df_model[
        [feature_1, feature_2]
    ]

    y = df_model[TARGET_COL]

    # t-SNE
    if usar_tsne:

        print("\nRodando t-SNE 2D...\n")

        scaler_original = StandardScaler()

        X_scaled_original = scaler_original.fit_transform(
            X_original
        )

        inicio_tsne = time.perf_counter()

        tsne = TSNE(
            n_components=2,
            perplexity=TSNE_PERPLEXITY,
            learning_rate=TSNE_LEARNING_RATE,
            early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
            early_exaggeration=TSNE_EARLY_EXAGGERATION,
            n_iter=TSNE_N_ITER,
            exaggeration=TSNE_EXAGGERATION,
            metric=TSNE_METRIC,
            initialization=TSNE_INITIALIZATION,
            negative_gradient_method=TSNE_NEGATIVE_GRADIENT_METHOD,
            n_jobs=TSNE_N_JOBS,
            random_state=estado_randomico,
            verbose=TSNE_VERBOSE
        )

        X_tsne = tsne.fit(X_scaled_original)
        X_tsne = np.asarray(X_tsne)

        fim_tsne = time.perf_counter()

        print(
            f"\nt-SNE 2D finalizado em "
            f"{(fim_tsne - inicio_tsne):.2f} segundos."
        )

        df_model["TSNE_1"] = X_tsne[:, 0]
        df_model["TSNE_2"] = X_tsne[:, 1]

        cols_modelo = ["TSNE_1", "TSNE_2"]

        titulo_corr = "Correlação Spearman - t-SNE 2D"
        titulo_scatter = "Distribuição 2D após t-SNE"
        titulo_prob = "Mapa de Probabilidade do GMM após t-SNE"
        titulo_relatorio = "Relatório GMM após t-SNE 2D (Spearman)"

    else:

        cols_modelo = [feature_1, feature_2]

        titulo_corr = "Correlação Spearman - Features Originais"
        titulo_scatter = "Distribuição 2D das Features Originais"
        titulo_prob = "Mapa de Probabilidade do GMM - Features Originais"
        titulo_relatorio = "Relatório GMM 2D - Features Originais"

    # FEATURES PARA GMM
    X = df_model[cols_modelo]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # GMM
    gmm = GaussianMixture(
        n_components=numero_de_componentes,
        covariance_type=tipo_matriz_covariancia,
        random_state=estado_randomico,
        reg_covar=erro_numerico,
        n_init=inicializacoes_gausianas
    )

    gmm.fit(X_scaled)

    # CLUSTERS
    clusters = gmm.predict(X_scaled)

    ct = pd.crosstab(clusters, y)

    print("\nTabela Cluster x Classe Real:")
    print(ct)

    if 1 not in ct.columns:
        raise ValueError("Nenhuma fraude encontrada nos clusters.")

    cluster_fraude = ct[1].idxmax()

    print(f"\nCluster identificado como fraude: {cluster_fraude}")

    # PROBABILIDADES
    score = gmm.predict_proba(X_scaled)[:, cluster_fraude]

    score = np.clip(
        score,
        1e-15,
        1 - 1e-15
    )

    # PREDIÇÃO
    y_pred = (score >= THRESHOLD).astype(int)

    # MÉTRICAS
    prec, rec, _ = precision_recall_curve(y, score)

    auc_pr = auc(rec, prec)

    mcc = matthews_corrcoef(y, y_pred)

    ks = ks_2samp(
        score[y == 0],
        score[y == 1]
    ).statistic

    ll = log_loss(y, score)

    score_final = calcular_score_final()

    # MATRIZ CONFUSÃO
    cm = confusion_matrix(
        y,
        y_pred,
        labels=[0, 1]
    )

    cm_percent = (
        cm.astype(float)
        / cm.sum(axis=1)[:, np.newaxis]
    ) * 100

    texto_cm = []

    for i in range(2):

        linha = []

        for j in range(2):

            linha.append(
                f"{cm_percent[i, j]:.2f}%"
                f"<br>({cm[i, j]})"
            )

        texto_cm.append(linha)

    # CORRELAÇÃO SPEARMAN
    corr = df_model[
        cols_modelo
    ].corr(method="spearman")

    # DATASETS SCATTER
    df_fraude = df_model[
        df_model[TARGET_COL] == 1
    ]

    df_nao_fraude = df_model[
        df_model[TARGET_COL] == 0
    ]

    # LIMITES COM FOLGA
    x_min = df_model[cols_modelo[0]].min()
    x_max = df_model[cols_modelo[0]].max()

    y_min = df_model[cols_modelo[1]].min()
    y_max = df_model[cols_modelo[1]].max()

    x_pad = (x_max - x_min) * 0.12
    y_pad = (y_max - y_min) * 0.12

    x_range = [
        x_min - x_pad,
        x_max + x_pad
    ]

    y_range = [
        y_min - y_pad,
        y_max + y_pad
    ]

    # FIGURA
    fig = make_subplots(
        rows=4,
        cols=2,

        specs=[
            [
                {"type": "heatmap"},
                {"type": "heatmap"}
            ],
            [
                {"colspan": 2},
                None
            ],
            [
                {"colspan": 2},
                None
            ],
            [
                {"colspan": 2},
                None
            ]
        ],

        row_heights=[
            0.22,
            0.13,
            0.32,
            0.33
        ],

        horizontal_spacing=0.12,
        vertical_spacing=0.10,

        subplot_titles=(
            titulo_corr,
            "Matriz de Confusão (%)",
            "",
            titulo_scatter,
            titulo_prob
        )
    )

    # HEATMAP CORRELAÇÃO
    fig.add_trace(
        go.Heatmap(
            z=corr.values,
            x=cols_modelo,
            y=cols_modelo,
            text=np.round(corr.values, 3),
            texttemplate="%{text}",
            textfont=dict(size=18),
            colorscale="RdBu",
            zmin=-1,
            zmax=1,
            showscale=False
        ),

        row=1,
        col=1
    )

    # HEATMAP CONFUSÃO
    fig.add_trace(
        go.Heatmap(
            z=cm_percent,
            x=[
                "Pred Não Fraude",
                "Pred Fraude"
            ],
            y=[
                "Real Não Fraude",
                "Real Fraude"
            ],
            text=texto_cm,
            texttemplate="%{text}",
            textfont=dict(size=18),
            colorscale="Blues",
            zmin=0,
            zmax=100,
            showscale=False
        ),

        row=1,
        col=2
    )

    # MÉTRICAS
    metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}<br>
Score Final: {score_final:.4f}
"""

    fig.add_trace(
        go.Scatter(
            x=[0.5],
            y=[0.5],
            mode="text",
            text=[metricas],
            textfont=dict(size=20),
            showlegend=False
        ),

        row=2,
        col=1
    )

    fig.update_xaxes(
        visible=False,
        range=[0, 1],
        row=2,
        col=1
    )

    fig.update_yaxes(
        visible=False,
        range=[0, 1],
        row=2,
        col=1
    )

    # SCATTER NÃO FRAUDE
    fig.add_trace(
        go.Scattergl(
            x=df_nao_fraude[cols_modelo[0]],
            y=df_nao_fraude[cols_modelo[1]],
            mode="markers",
            name="Não Fraude",
            marker=dict(
                color="rgba(0,0,255,0.22)",
                size=5
            )
        ),

        row=3,
        col=1
    )

    # SCATTER FRAUDE
    fig.add_trace(
        go.Scattergl(
            x=df_fraude[cols_modelo[0]],
            y=df_fraude[cols_modelo[1]],
            mode="markers",
            name="Fraude",
            marker=dict(
                color="rgba(255,0,0,0.92)",
                size=5
            )
        ),

        row=3,
        col=1
    )

    # GRID PARA MAPA DE PROBABILIDADE
    grid_size = 250

    x_grid = np.linspace(
        x_range[0],
        x_range[1],
        grid_size
    )

    y_grid = np.linspace(
        y_range[0],
        y_range[1],
        grid_size
    )

    xx, yy = np.meshgrid(
        x_grid,
        y_grid
    )

    grid_points = np.c_[
        xx.ravel(),
        yy.ravel()
    ]

    grid_points_df = pd.DataFrame(
        grid_points,
        columns=cols_modelo
    )

    grid_scaled = scaler.transform(
        grid_points_df
    )

    prob_grid = gmm.predict_proba(
        grid_scaled
    )[:, cluster_fraude]

    prob_grid = prob_grid.reshape(
        xx.shape
    )

    # HEATMAP DE PROBABILIDADE GMM
    fig.add_trace(
        go.Heatmap(
            x=x_grid,
            y=y_grid,
            z=prob_grid,

            colorscale="Turbo",

            zmin=0,
            zmax=1,

            opacity=0.82,

            colorbar=dict(
                title=dict(
                    text="Probabilidade de Fraude",
                    side="bottom",
                    font=dict(size=16)
                ),
                orientation="h",
                x=0.5,
                xanchor="center",
                y=-0.10,
                yanchor="top",
                len=0.70,
                thickness=22,
                tickfont=dict(size=14)
            ),

            name="Probabilidade GMM",
            showscale=True
        ),

        row=4,
        col=1
    )

    # CONTORNO DO THRESHOLD
    fig.add_trace(
        go.Contour(
            x=x_grid,
            y=y_grid,
            z=prob_grid,

            contours=dict(
                start=THRESHOLD,
                end=THRESHOLD,
                size=1,
                coloring="none",
                showlabels=True
            ),

            line=dict(
                color="black",
                width=3
            ),

            showscale=False,

            name=f"Threshold {THRESHOLD:.2f}"
        ),

        row=4,
        col=1
    )

    # PONTOS NÃO FRAUDE SOBRE O MAPA
    fig.add_trace(
        go.Scattergl(
            x=df_nao_fraude[cols_modelo[0]],
            y=df_nao_fraude[cols_modelo[1]],

            mode="markers",

            name="Não Fraude - mapa",

            marker=dict(
                color="rgba(255,255,255,0.35)",
                size=4,
                line=dict(
                    color="rgba(0,0,255,0.35)",
                    width=0.5
                )
            ),

            showlegend=False
        ),

        row=4,
        col=1
    )

    # PONTOS FRAUDE SOBRE O MAPA
    fig.add_trace(
        go.Scattergl(
            x=df_fraude[cols_modelo[0]],
            y=df_fraude[cols_modelo[1]],

            mode="markers",

            name="Fraude - mapa",

            marker=dict(
                color="rgba(255,0,0,0.95)",
                size=6,
                line=dict(
                    color="white",
                    width=0.5
                )
            ),

            showlegend=False
        ),

        row=4,
        col=1
    )

    # AXIS SCATTER
    fig.update_xaxes(
        title_text=cols_modelo[0],
        title_font=dict(size=22),
        tickfont=dict(size=15),
        range=x_range,
        row=3,
        col=1
    )

    fig.update_yaxes(
        title_text=cols_modelo[1],
        title_font=dict(size=22),
        tickfont=dict(size=15),
        range=y_range,
        row=3,
        col=1
    )

    # AXIS MAPA PROBABILIDADE
    fig.update_xaxes(
        title_text=cols_modelo[0],
        title_font=dict(size=22),
        tickfont=dict(size=15),
        range=x_range,
        row=4,
        col=1
    )

    fig.update_yaxes(
        title_text=cols_modelo[1],
        title_font=dict(size=22),
        tickfont=dict(size=15),
        range=y_range,
        row=4,
        col=1
    )

    # LAYOUT
    if usar_tsne:
        descricao_features = f"""
        Features originais:
        {feature_1} vs {feature_2}
        <br>
        Novas features:
        TSNE_1 vs TSNE_2
        """
    else:
        descricao_features = f"""
        Features originais:
        {feature_1} vs {feature_2}
        """

    fig.update_layout(
        title=dict(
            text=f"""
            {titulo_relatorio}
            <br>
            Rank {RANK}
            <br>
            {descricao_features}
            <br>
            Base completa: 100% fraudes + 100% não fraudes
            <br>
            Corte: Probabilidade Cluster Fraude ≥ {THRESHOLD:.2f}
            """,

            x=0.5,
            y=0.985,

            xanchor="center",
            yanchor="top",

            font=dict(size=26)
        ),

        width=1900,
        height=2750,

        template="plotly_white",

        font=dict(size=18),

        margin=dict(
            t=380,
            b=280,
            l=120,
            r=120
        ),

        legend=dict(
            orientation="h",
            font=dict(size=18),
            yanchor="bottom",
            y=0.015,
            xanchor="center",
            x=0.5
        )
    )

    # SAVE HTML
    fig.write_html(
        HTML_PATH,
        include_plotlyjs="cdn"
    )

    print("\nHTML GERADO COM SUCESSO:")
    print(HTML_PATH)

    return {
        "HTML_PATH": HTML_PATH,
        "Rank": RANK,
        "Tipo": tipo_relatorio,
        "Features": [feature_1, feature_2],
        "AUC_PR": auc_pr,
        "MCC": mcc,
        "KS": ks,
        "Log_Loss": ll,
        "Score_Final": score_final
    }

## 1ST COLOCADO

### ORIGINAL 

In [35]:
resultado_2d_original = gerar_relatorio_2d(
    RANK=1,
    usar_tsne=False,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,
    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas=inicializacoes_gausianas_D,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D,
    erro_numerico=erro_numerico_D,
    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_2d_original)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 1
Features: V11 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              278899   71
1                5416  421

Cluster identificado como fraude: 1

HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\2d_rank_1_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\2d_rank_1_orig.html', 'Rank': 1, 'Tipo': 'orig', 'Features': ['V11', 'V17'], 'AUC_PR': 0.5791002707773756, 'MCC': 0.24521576046382343, 'KS': np.float64(0.8570377754320075), 'Log_Loss': 0.12106141297953486, 'Score_Final': 0.737689}


### T-SNE

In [ ]:
resultado_2d_tsne = gerar_relatorio_2d(
    RANK=1,
    usar_tsne=True,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,
    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas=inicializacoes_gausianas_D,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D,
    erro_numerico=erro_numerico_D,
    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_2d_tsne)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 1
Features: V11 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 2D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_iter=1500, n_jobs=7, negative_gradient_method='bh', random_state=42,
     verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...


## 2ND COLOCADO

### ORIGINAL 

In [ ]:
resultado_2d_original = gerar_relatorio_2d(
    RANK=2,
    usar_tsne=False,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,
    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas=inicializacoes_gausianas_D,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D,
    erro_numerico=erro_numerico_D,
    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_2d_original)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 2
Features: V15 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              280486  115
1                3829  377

Cluster identificado como fraude: 1

HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\2d_rank_2_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\2d_rank_2_orig.html', 'Rank': 2, 'Tipo': 'orig', 'Features': ['V15', 'V17'], 'AUC_PR': 0.5335491356770402, 'MCC': 0.25916634191193516, 'KS': np.float64(0.7930215384316234), 'Log_Loss': 0.09086939112498978, 'Score_Final': 0.718213}


### T-SNE

In [ ]:
resultado_2d_tsne = gerar_relatorio_2d(
    RANK=2,
    usar_tsne=True,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,
    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas=inicializacoes_gausianas_D,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D,
    erro_numerico=erro_numerico_D,
    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_2d_tsne)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 2
Features: V15 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 2D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_iter=1500, n_jobs=7, negative_gradient_method='bh', random_state=42,
     verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 124.97 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 36.94 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.07 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   50, KL divergence 6.244

## 3RD COLOCADO

### ORIGINAL 

In [ ]:
resultado_2d_original = gerar_relatorio_2d(
    RANK=3,
    usar_tsne=False,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,
    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas=inicializacoes_gausianas_D,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D,
    erro_numerico=erro_numerico_D,
    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_2d_original)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 3
Features: V4 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              272763   53
1               11552  439

Cluster identificado como fraude: 1

HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\2d_rank_3_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\2d_rank_3_orig.html', 'Rank': 3, 'Tipo': 'orig', 'Features': ['V4', 'V17'], 'AUC_PR': 0.5675965017428086, 'MCC': 0.17610781932949898, 'KS': np.float64(0.8554901961625353), 'Log_Loss': 0.16614986551451374, 'Score_Final': 0.717166}


### T-SNE

In [ ]:
resultado_2d_tsne = gerar_relatorio_2d(
    RANK=3,
    usar_tsne=True,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,
    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas=inicializacoes_gausianas_D,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D,
    erro_numerico=erro_numerico_D,
    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_2d_tsne)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 3
Features: V4 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 2D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_iter=1500, n_jobs=7, negative_gradient_method='bh', random_state=42,
     verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 194.06 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 52.42 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.10 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   50, KL divergence 6.2507

KeyboardInterrupt: 

## 4TH COLOCADO

### ORIGINAL 

In [ ]:
resultado_2d_original = gerar_relatorio_2d(
    RANK=4,
    usar_tsne=False,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,
    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas=inicializacoes_gausianas_D,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D,
    erro_numerico=erro_numerico_D,
    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_2d_original)

### T-SNE

In [ ]:
resultado_2d_tsne = gerar_relatorio_2d(
    RANK=4,
    usar_tsne=True,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,
    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas=inicializacoes_gausianas_D,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D,
    erro_numerico=erro_numerico_D,
    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_2d_tsne)

## 5TH COLOCADO

### ORIGINAL 

In [ ]:
resultado_2d_original = gerar_relatorio_2d(
    RANK=5,
    usar_tsne=False,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,
    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas=inicializacoes_gausianas_D,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D,
    erro_numerico=erro_numerico_D,
    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_2d_original)

### T-SNE

In [ ]:
resultado_2d_tsne = gerar_relatorio_2d(
    RANK=5,
    usar_tsne=True,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,
    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas=inicializacoes_gausianas_D,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D,
    erro_numerico=erro_numerico_D,
    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_2d_tsne)